# Gold Layer — Business Aggregations & Feature Engineering

**Medallion Stage:** Gold (Business-Ready)

The gold layer transforms clean silver data into purpose-built analytical tables.
Every table is exported to both Parquet (for Python) and CSV (for Power BI import).

**Tables produced:**

| Table | Description |
|---|---|
| `attrition_by_department` | Attrition KPIs per department |
| `attrition_by_role` | Attrition by dept + job role combination |
| `attrition_by_demographics` | Attrition across age/marital/education/distance |
| `satisfaction_analysis` | Attrition rate per satisfaction score (all 4 dimensions) |
| `salary_analysis` | Income distribution by level + role |
| `tenure_analysis` | Attrition by years at company (reveals the 3-year crisis) |
| `overtime_impact` | Overtime × job satisfaction interaction |
| `risk_scores` | Per-employee composite flight risk score |
| `executive_summary` | Top-level KPI table for exec dashboards |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('Set2')

silver_path = ROOT / 'data' / 'silver' / 'employee_attrition_silver.parquet'
df = pd.read_parquet(silver_path)
print(f'Silver loaded: {df.shape}')
print(f'Attrition rate: {df["Attrition"].mean():.1%}')

## 1. Run the Gold Aggregator

In [ ]:
from src.utils.config import load_config
from src.gold.aggregate import GoldAggregator

cfg = load_config()
agg = GoldAggregator(cfg)
tables = agg.build(df)

print('Gold tables built:')
for name, tbl in tables.items():
    print(f'  {name:<35}  {tbl.shape[0]:>4} rows  x  {tbl.shape[1]} cols')

## 2. Attrition by Department

In [ ]:
dept = tables['attrition_by_department'].sort_values('AttritionRate', ascending=False)
display(dept)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(dept['Department'], dept['AttritionRate'] * 100,
            color=['#e74c3c' if r > 0.15 else '#3498db' for r in dept['AttritionRate']])
axes[0].axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', label='Company avg')
axes[0].set_title('Attrition Rate by Department', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].legend()

axes[1].bar(dept['Department'], dept['OvertimeRate'] * 100, color='#e67e22')
axes[1].set_title('Overtime Rate by Department', fontweight='bold')
axes[1].set_ylabel('Employees Working Overtime (%)')

plt.tight_layout()
plt.show()

## 3. Attrition by Job Role

In [ ]:
role = tables['attrition_by_role'].sort_values('AttritionRate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(role['JobRole'], role['AttritionRate'] * 100,
               color=['#e74c3c' if r > 0.20 else '#3498db' for r in role['AttritionRate']])
ax.axvline(df['Attrition'].mean() * 100, color='gray', linestyle='--', linewidth=1.5, label='Company avg')
for bar, val in zip(bars, role['AttritionRate']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=9)
ax.set_title('Attrition Rate by Job Role', fontweight='bold', fontsize=13)
ax.set_xlabel('Attrition Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Satisfaction → Attrition Relationship

In [ ]:
sat = tables['satisfaction_analysis']
dimensions = sat['SatisfactionDimension'].unique()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, dim in enumerate(dimensions):
    sub = sat[sat['SatisfactionDimension'] == dim].sort_values('Score')
    axes[i].bar(sub['Score'], sub['AttritionRate'] * 100,
                color=[f'#{max(0, int(r*255)):02x}50{max(0, int((1-r)*200)):02x}' for r in sub['AttritionRate']])
    axes[i].set_title(dim, fontweight='bold')
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Attrition Rate (%)')
    axes[i].set_xticks(sub['Score'])

plt.suptitle('Attrition Rate by Satisfaction Dimension', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Tenure Analysis — The 3-Year Crisis

In [ ]:
tenure = tables['tenure_analysis']

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.fill_between(tenure['YearsAtCompany'], tenure['AttritionRate'] * 100,
                 alpha=0.4, color='#e74c3c', label='Attrition Rate')
ax1.plot(tenure['YearsAtCompany'], tenure['AttritionRate'] * 100,
         color='#e74c3c', linewidth=2)
ax1.set_ylabel('Attrition Rate (%)', color='#e74c3c')
ax1.set_xlabel('Years at Company')

ax2.bar(tenure['YearsAtCompany'], tenure['TotalEmployees'],
        alpha=0.3, color='steelblue', label='Employee Count')
ax2.set_ylabel('Employee Count', color='steelblue')

ax1.set_title('Attrition Rate vs. Tenure — The 3-Year Crisis Revealed', fontweight='bold', fontsize=13)
ax1.axvspan(0.5, 3.5, alpha=0.1, color='red', label='High-risk window (years 1–3)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

print('Key finding: Attrition is highest in the first 1-3 years (onboarding failure)')
print('Secondary peak at 5-8 years (growth plateau effect)')

## 6. Flight Risk Scores — Executive View

In [ ]:
risk = tables['risk_scores']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk band distribution
band_counts = risk['RiskBand'].value_counts().reindex(['Low', 'Moderate', 'High', 'Critical'])
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
axes[0].bar(band_counts.index, band_counts.values, color=colors, edgecolor='white')
for i, (label, val) in enumerate(band_counts.items()):
    pct = val / len(risk)
    axes[0].text(i, val + 5, f'{val}\n({pct:.1%})', ha='center', va='bottom', fontsize=9)
axes[0].set_title('Employees by Flight Risk Band', fontweight='bold')
axes[0].set_ylabel('Count')

# Risk score distribution by actual attrition
axes[1].hist([risk.loc[risk['Attrition']==0, 'FlightRiskScore'],
              risk.loc[risk['Attrition']==1, 'FlightRiskScore']],
             bins=20, stacked=True, color=['#2ecc71', '#e74c3c'],
             label=['Stayed', 'Left'], edgecolor='white')
axes[1].set_title('Flight Risk Score Distribution\nby Actual Attrition', fontweight='bold')
axes[1].set_xlabel('Flight Risk Score')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary stats
print('\nMean risk score by attrition outcome:')
print(risk.groupby('Attrition')['FlightRiskScore'].mean().rename({0: 'Stayed', 1: 'Left'}))

## 7. Executive Summary Table

In [ ]:
summary = tables['executive_summary']
display(summary.style.set_properties(**{'text-align': 'left'}).set_caption('Executive KPI Summary'))

print(f'\nAll gold tables exported to: {ROOT / "data" / "exports"}')
print('Connect Power BI to the exports/ folder via Get Data → Text/CSV')

---
## Gold Layer Summary

9 analytical tables built and exported to `data/exports/` as CSV files.

**Power BI connection:** Open Power BI Desktop → Get Data → Folder → point to `data/exports/`

**Next step:** `04_exploratory_data_analysis.ipynb` — deep statistical analysis